# Chapter 5 - Upgrade Semantic Views

Move from a thin semantic view to curated base + ontology semantic views.

## Learn
- How explicit metric definitions reduce ambiguity
- Why separate base and ontology models help routing


In [ ]:
-- Curated base semantic view with explicit revenue metric
CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML(
  'SNOWFLAKE_EXAMPLE.SEMANTIC_MODELS',
  $$
name: SV_SAM_DRIFT_BASE
description: Curated base semantic model for Drift source tables
module_custom_instructions: |
  Always define revenue as SUM(invoice_line.unit_price * invoice_line.quantity).
  Never estimate revenue from track.unit_price alone.
tables:
  - name: SNOWFLAKE_EXAMPLE.SAM_DRIFT.CUSTOMER
  - name: SNOWFLAKE_EXAMPLE.SAM_DRIFT.EMPLOYEE
  - name: SNOWFLAKE_EXAMPLE.SAM_DRIFT.INVOICE
  - name: SNOWFLAKE_EXAMPLE.SAM_DRIFT.INVOICE_LINE
  - name: SNOWFLAKE_EXAMPLE.SAM_DRIFT.TRACK
  - name: SNOWFLAKE_EXAMPLE.SAM_DRIFT.GENRE
verified_queries:
  - name: yoy_jazz_vs_rock
    question: How did jazz revenue trend year over year from 2020 to 2024, compared to rock?
    sql: |
      SELECT EXTRACT(year FROM i.invoice_date) AS year, g.name AS genre,
             SUM(il.unit_price * il.quantity) AS revenue
      FROM SNOWFLAKE_EXAMPLE.SAM_DRIFT.invoice i
      JOIN SNOWFLAKE_EXAMPLE.SAM_DRIFT.invoice_line il ON i.invoice_id = il.invoice_id
      JOIN SNOWFLAKE_EXAMPLE.SAM_DRIFT.track t ON il.track_id = t.track_id
      JOIN SNOWFLAKE_EXAMPLE.SAM_DRIFT.genre g ON t.genre_id = g.genre_id
      WHERE g.name IN ('Jazz', 'Rock')
      GROUP BY 1,2
      ORDER BY 1,2
$$,
  FALSE
);


In [ ]:
-- Ontology semantic view over abstract views
CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML(
  'SNOWFLAKE_EXAMPLE.SEMANTIC_MODELS',
  $$
name: SV_SAM_DRIFT_ONTOLOGY
description: Ontology-centric semantic model for Drift abstractions
tables:
  - name: SNOWFLAKE_EXAMPLE.SAM_DRIFT.VW_ONT_PERSON
  - name: SNOWFLAKE_EXAMPLE.SAM_DRIFT.VW_ONT_MEDIAENTITY
  - name: SNOWFLAKE_EXAMPLE.SAM_DRIFT.VW_ONT_SALE
verified_queries:
  - name: person_coverage
    question: For every country, compare customer and employee coverage.
    sql: |
      SELECT country, person_type, COUNT(*) AS people
      FROM SNOWFLAKE_EXAMPLE.SAM_DRIFT.vw_ont_person
      GROUP BY 1,2
      ORDER BY 1,2
$$,
  FALSE
);


In [ ]:
-- Check
SHOW SEMANTIC VIEWS IN SCHEMA SNOWFLAKE_EXAMPLE.SEMANTIC_MODELS LIKE 'SV_SAM_DRIFT%';


## Reflect
The base model handles precise operational questions; the ontology model handles abstraction and cross-entity reasoning.

## Next
Open [`ch06_rebuild_and_compare.ipynb`](ch06_rebuild_and_compare.ipynb).
